<a href="https://www.kaggle.com/code/aleksandrmorozov123/ai-agents?scriptVersionId=246762604" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Implementation of reflection patterns on Groq models**

In [2]:
! pip install groq 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 6.0 MB/s eta 0:00:00


In [3]:
!pip install python-dotenv

In [4]:
import os
import re
import math
import json
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Groq")

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq(api_key=secret_value_0)

In [5]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

In [6]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Bucket Sort algorithm"
    }
)

In [7]:
bucketsort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": bucketsort_code
    }
)

In [8]:
display_markdown(bucketsort_code, raw=True)

Here is a Python implementation of the Bucket Sort algorithm:
```
def bucket_sort(arr):
    # Create empty buckets
    buckets = [[] for _ in range(len(arr))]

    # Assign elements to their corresponding buckets
    for elem in arr:
        index = int(len(buckets) * elem)
        buckets[index].append(elem)

    # Sort each bucket individually
    for bucket in buckets:
        bucket.sort()

    # Concatenate the sorted buckets
    sorted_arr = []
    for bucket in buckets:
        sorted_arr.extend(bucket)

    return sorted_arr
```
Here's an explanation of how the implementation works:

1. We create an array of empty buckets, where the number of buckets is equal to the length of the input array.
2. We iterate through the input array and assign each element to its corresponding bucket based on its value. We use the formula `int(len(buckets) * elem)` to determine the index of the bucket where the element should be placed. This ensures that elements with similar values are grouped together in the same bucket.
3. We sort each bucket individually using the built-in `sort()` method. This is because bucket sort is a distribution sort, and we can use any sorting algorithm to sort the elements within each bucket.
4. Finally, we concatenate the sorted buckets to produce the final sorted array.

Note that this implementation assumes that the input array consists of elements in the range [0, 1). If the input array has a different range, you may need to adjust the bucket assignment step accordingly.

Example usage:
```
arr = [0.4, 0.2, 0.1, 0.3, 0.5, 0.6]
sorted_arr = bucket_sort(arr)
print(sorted_arr)  # [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
```
Let me know if you have any questions or if you'd like me to revise anything!

In [9]:
# reflection part
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are an experienced phD computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": bucketsort_code
    }
)

In [11]:
# generate critique to the python code
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [12]:
display_markdown(critique, raw=True)

Overall, your implementation of the Bucket Sort algorithm is well-structured and easy to understand. Here's a critique of your code and some recommendations for improvement:

**Critique:**

1. **Magic number:** The formula `int(len(buckets) * elem)` used to determine the index of the bucket might be difficult to understand for someone who is not familiar with the Bucket Sort algorithm. It would be better to introduce a variable, e.g., `bucket_size`, to make the code more readable.

2. **Inconsistent naming conventions:** Python's PEP 8 style guide recommends using underscore-separated names for variables and functions. You've used camelCase for `bucket_sort`, which is more commonly used in languages like Java. Try to stick to a single convention throughout your code.

3. **Unnecessary memory allocation:** Creating an array of empty buckets with a size equal to the length of the input array might be inefficient for large input arrays, especially if the input array contains a lot of duplicate values. You could use a list comprehension to create buckets only as needed.

4. **Type hinting and documentation:** While you've provided an explanation of how the code works, it would be beneficial to include type hints and docstrings in your code to make it more readable and maintainable.

**Recommendations:**

Here's an updated version of your code with the above recommendations in mind:

```
def bucket_sort(arr: list[float]) -> list[float]:
    """
    Sorts an array of floats in the range [0, 1) using the Bucket Sort algorithm.

    Args:
        arr: The input array to be sorted.

    Returns:
        A sorted array of floats.
    """
    # Calculate the size of each bucket
    bucket_size = len(arr)

    # Create a list to store the buckets
    buckets = []

    # Assign elements to their corresponding buckets
    for elem in arr:
        bucket_index = int(bucket_size * elem)
        while len(buckets) <= bucket_index:
            buckets.append([])
        buckets[bucket_index].append(elem)

    # Sort each bucket individually
    for bucket in buckets:
        bucket.sort()

    # Concatenate the sorted buckets
    sorted_arr = []
    for bucket in buckets:
        sorted_arr.extend(bucket)

    return sorted_arr

# Example usage:
arr = [0.4, 0.2, 0.1, 0.3, 0.5, 0.6]
sorted_arr = bucket_sort(arr)
print(sorted_arr)  # [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
```

The updated code includes type hints, docstrings, and a more readable bucket assignment step. It also creates buckets dynamically to reduce memory allocation for large input arrays.

In [13]:
# add critique to the generation chat history
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [14]:
# generation step 2
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [15]:
display_markdown(essay, raw=True)

Thank you so much for the detailed critique and recommendations! I really appreciate it. You've provided some excellent suggestions that improve the code's readability, maintainability, and efficiency.

I'm glad you found the implementation well-structured and easy to understand. Here's a response to each of your points:

**Magic number:**
You're absolutely right. Introducing a variable like `bucket_size` makes the code more readable and easier to understand.

**Inconsistent naming conventions:**
I apologize for the mistake. You're correct that I used camelCase for the function name, which is not consistent with Python's PEP 8 style guide. I'll make sure to use underscore-separated names for variables and functions going forward.

**Unnecessary memory allocation:**
You've made a great point about the inefficient memory allocation. Creating buckets dynamically as needed is a much better approach, especially for large input arrays.

**Type hinting and documentation:**
I completely agree that type hints and docstrings are essential for making the code more readable and maintainable. I'll make sure to include them in my code going forward.

I love the updated version of the code you provided! It's much more readable and efficient. The use of type hints, docstrings, and dynamic bucket creation are all great improvements. The code is now more Pythonic and follows the PEP 8 style guide.

Thanks again for your feedback and recommendations. I'll keep them in mind when writing future code.

**Implement tool function for agent**

In [16]:
import json

def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Miami, Florida
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Miami":
		return json.dumps({"Fahrenheit": 75, "unit": unit})

	else:
		return json.dumps({"temperature": 130, "unit": unit})

In [17]:
get_current_weather(location="Miami", unit="Fahrenheit")

'{"Fahrenheit": 75, "unit": "Fahrenheit"}'

In [18]:
MODEL = "llama-3.3-70b-versatile"
GROQ_CLIENT = Groq(api_key=secret_value_0)

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [19]:
# ask the question
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Miami, in Fahrenheit?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = GROQ_CLIENT.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<tool_call>
{"name": "get_current_weather", "arguments": {"location": "Miami", "unit": "fahrenheit"}}
</tool_call>


In [20]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

In [21]:
parsed_output = parse_tool_call_str(output)
parsed_output

{'name': 'get_current_weather',
 'arguments': {'location': 'Miami', 'unit': 'fahrenheit'}}

In [22]:
result = get_current_weather(**parsed_output["arguments"])

In [23]:
result

'{"Fahrenheit": 75, "unit": "fahrenheit"}'

In [24]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

In [25]:
GROQ_CLIENT.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Miami is 75°F.'

**Implementation of planning pattern**

In [26]:
!pip install agents

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.6/118.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 27.8 MB/s eta 0:00:00
  Created wheel for agents: filename=agents-1.4.0-py3-none-any.whl size=62714 sha256=de185e71e3cfa54ced6d72403421670dda4fb9860e4ba33c86bf675fa1564a05
  Stored in directory: /root/.cache/pip/wheels/6a/41/e1/4d212ede7c9752d8f7050d78fa3295304bc22b8fb530d9ebe4
Successfully built agents


In [27]:
!pip install agentic_patterns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 0.28.0
    Uninstalling groq-0.28.0:
      Successfully uninstalled groq-0.28.0


In [28]:
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content
MODEL = "llama-3.3-70b-versatile"

In [29]:
# define the system prompt as a constant
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
%s
</tools>

Example session:

<question>What's the current temperature in Miami?</question>
<thought>I need to get the current weather in Miami</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 75, "unit": "fahrenheit"}}</observation>

You then output:

<response>The current temperature in Miami is 75 degrees Fahrenheit</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""

In [30]:
# defining the tools
@tool
def sum_two_elements(a: int, b: int) -> int:
    """
    Computes the sum of two integers.

    Args:
        a (int): The first integer to be summed.
        b (int): The second integer to be summed.

    Returns:
        int: The sum of `a` and `b`.
    """
    return a + b


@tool
def multiply_two_elements(a: int, b: int) -> int:
    """
    Multiplies two integers.

    Args:
        a (int): The first integer to multiply.
        b (int): The second integer to multiply.

    Returns:
        int: The product of `a` and `b`.
    """
    return a * b

@tool
def compute_log(x: int) -> float | str:
    """
    Computes the logarithm of an integer `x` with an optional base.

    Args:
        x (int): The integer value for which the logarithm is computed. Must be greater than 0.

    Returns:
        float: The logarithm of `x` to the specified `base`.
    """
    if x <= 0:
        return "Logarithm is undefined for values less than or equal to 0."
    
    return math.log(x)


available_tools = {
    "sum_two_elements": sum_two_elements,
    "multiply_two_elements": multiply_two_elements,
    "compute_log": compute_log
}

In [31]:
# check the tool operator
print("Tool name: ", sum_two_elements.name)
print("Tool signature: ", sum_two_elements.fn_signature)

Tool name:  sum_two_elements
Tool signature:  {"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}}
